# 05 - Backtesting: Donchian Breakout + Trailing Stop

**Capítulo**: 05 - Breakout

**Objetivo**: Backtest de breakout Donchian con trailing stop ATR y análisis de position sizing.

---

In [ ]:
import sys
sys.path.insert(0, '../..')

import pandas as pd
import numpy as np
from backtesting import Backtest, Strategy

from curso.lib.data import download_historical
from curso.lib.backtest import run_backtest, extract_metrics, metrics_to_dataframe
from curso.lib.reporting import plot_equity_curve, plot_drawdown, print_metrics_table

import matplotlib.pyplot as plt
plt.style.use('seaborn-v0_8-whitegrid')
print('Setup completado ✓')

## 1. Datos

In [ ]:
TICKER = 'GLD'
df = download_historical(TICKER)
print(f'{TICKER}: {len(df)} registros')

## 2. Estrategia Donchian Breakout

In [ ]:
def Donchian_Upper(high, n):
    return pd.Series(high).rolling(n).max().values

def Donchian_Lower(low, n):
    return pd.Series(low).rolling(n).min().values

def ATR_calc(high, low, close, n):
    h, l, c = pd.Series(high), pd.Series(low), pd.Series(close)
    tr = pd.concat([h-l, (h-c.shift()).abs(), (l-c.shift()).abs()], axis=1).max(axis=1)
    return tr.rolling(n).mean().values


class DonchianBreakout(Strategy):
    entry_period = 20
    exit_period = 10
    atr_period = 14
    atr_trailing = 3.0
    
    def init(self):
        self.dc_upper = self.I(Donchian_Upper, self.data.High, self.entry_period)
        self.dc_lower_exit = self.I(Donchian_Lower, self.data.Low, self.exit_period)
        self.atr = self.I(ATR_calc, self.data.High, self.data.Low, self.data.Close, self.atr_period)
        self.highest_since_entry = 0
    
    def next(self):
        price = self.data.Close[-1]
        
        if not self.position:
            # Entry: breakout above upper channel
            if price > self.dc_upper[-2]:  # Compare to previous day's channel
                self.buy()
                self.highest_since_entry = price
        else:
            # Update trailing high
            self.highest_since_entry = max(self.highest_since_entry, price)
            
            # Trailing stop: highest - ATR*multiplier
            trailing_stop = self.highest_since_entry - self.atr_trailing * self.atr[-1]
            
            # Exit if price drops below trailing stop OR below exit channel
            if price < trailing_stop or price < self.dc_lower_exit[-1]:
                self.position.close()
                self.highest_since_entry = 0

## 3. Backtest

In [ ]:
stats, bt = run_backtest(df, DonchianBreakout)
metrics = extract_metrics(stats)
print_metrics_table(metrics_to_dataframe(metrics))

fig = plot_equity_curve(stats, title=f'{TICKER} - Donchian Breakout')
plt.show()

fig = plot_drawdown(stats, title=f'{TICKER} - Drawdown')
plt.show()

## 4. Análisis del trailing stop

In [ ]:
# Comparar diferentes multiplicadores de ATR
results = []
for mult in [2.0, 2.5, 3.0, 3.5, 4.0]:
    class TestStrat(DonchianBreakout):
        atr_trailing = mult
    s, _ = run_backtest(df, TestStrat)
    results.append({'ATR Mult': mult, 'Return %': s['Return [%]'], 
                    'Sharpe': s['Sharpe Ratio'], 'Max DD %': s['Max. Drawdown [%]'],
                    'Trades': s['# Trades']})

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

## 5. Conclusiones

In [ ]:
print('''
CONCLUSIONES
============
1. El breakout [funciona/no funciona] en este activo
2. Trailing stop óptimo: [multiplicador ATR]
3. Win rate bajo pero ratio win/loss: [evaluar]
4. Drawdown: [evaluar si < 25% target]
5. DECISIÓN: [aprobar / iterar / descartar]
''')